# EDA danger incendie — Fourcasters

Ce notebook explore les données **Météo des forêts** utilisées dans Fourcasters.

Le jeu de données contient le **niveau de danger prévu par Météo-France** pour chaque département, à J+1 et J+2.
Il ne s'agit pas du nombre de feux réellement observés.


## Préparation


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.cloud import bigquery

from fourcasters_dbt.configuration import (
    PROJET_GCP,
    DATASET_ANALYSE,
    configurer_google_cloud,
)

configurer_google_cloud()

DATASET = DATASET_ANALYSE
client = bigquery.Client(project=PROJET_GCP)

def lire_requete(sql):
    return client.query(sql).to_dataframe(create_bqstorage_client=False)


La connexion utilise les tables dbt du projet dans BigQuery. Les analyses restent donc basées sur les données utilisées par Power BI et le modèle.


## 1. Taille du jeu de données et période disponible


In [ ]:
resume = lire_requete(f"""
SELECT
  COUNT(*) AS lignes,
  COUNT(DISTINCT date_publication) AS jours_publication,
  COUNT(DISTINCT reference_time) AS publications,
  COUNT(DISTINCT numero_departement) AS departements,
  MIN(date_publication) AS date_debut,
  MAX(date_publication) AS date_fin
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
""")

resume


On vérifie ici la période réellement disponible et le nombre de départements couverts. La collecte Météo-France concerne les 96 départements métropolitains du projet.


## 2. Qualité et unicité des données


In [ ]:
qualite = lire_requete(f"""
SELECT
  COUNT(*) AS lignes,
  COUNT(DISTINCT id_danger_incendie) AS ids_uniques,
  COUNTIF(numero_departement IS NULL) AS departements_manquants,
  COUNTIF(niveau_danger IS NULL) AS niveaux_manquants,
  COUNTIF(niveau_danger NOT BETWEEN 1 AND 4) AS niveaux_invalides
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
""")

qualite


Le nombre d'identifiants uniques doit correspondre au nombre de lignes et les niveaux doivent rester compris entre 1 et 4. C'est le contrôle de base avant toute analyse.


## 3. Couverture des publications


In [ ]:
couverture = lire_requete(f"""
SELECT
  reference_time,
  echeance,
  COUNT(DISTINCT numero_departement) AS departements
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY reference_time, echeance
ORDER BY reference_time, echeance
""")

display(couverture["departements"].describe())

incompletes = couverture[couverture["departements"] != 96]
print("Nombre de publications incomplètes :", len(incompletes))
display(incompletes.tail(15))


Une publication complète contient 96 départements pour J1 et 96 pour J2. Une valeur différente mérite d'être vérifiée avant de comparer les niveaux de danger.


## 4. Répartition générale des niveaux de danger


In [ ]:
repartition = lire_requete(f"""
SELECT
  niveau_danger,
  COUNT(*) AS observations,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY niveau_danger
ORDER BY niveau_danger
""")

display(repartition)

repartition.plot(
    x="niveau_danger",
    y="pct",
    kind="bar",
    figsize=(7, 4),
    legend=False,
)
plt.title("Répartition des niveaux de danger")
plt.ylabel("Part des observations (%)")
plt.xlabel("Niveau de danger")
plt.show()


Cette répartition montre si les quatre niveaux sont équilibrés. Les niveaux faibles et modérés sont généralement plus fréquents que le niveau 4, ce qui est important pour le Machine Learning.


## 5. Différence entre les prévisions J1 et J2


In [ ]:
par_echeance = lire_requete(f"""
SELECT
  echeance,
  niveau_danger,
  COUNT(*) AS observations,
  ROUND(
    100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY echeance),
    2
  ) AS pct
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY echeance, niveau_danger
ORDER BY echeance, niveau_danger
""")

display(par_echeance)

table_echeance = par_echeance.pivot(
    index="niveau_danger",
    columns="echeance",
    values="pct",
)

table_echeance.plot(kind="bar", figsize=(8, 4))
plt.title("Niveaux de danger à J1 et J2")
plt.ylabel("Part des observations (%)")
plt.xlabel("Niveau de danger")
plt.show()


J1 et J2 devraient avoir des profils assez proches, mais pas forcément identiques. Les écarts montrent comment le danger prévu évolue quand l'horizon de prévision s'allonge.


## 6. Évolution du danger au fil de la saison


In [ ]:
evolution = lire_requete(f"""
SELECT
  date_publication,
  echeance,
  ROUND(AVG(niveau_danger), 2) AS niveau_moyen,
  COUNTIF(niveau_danger >= 3) AS departements_danger_eleve
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY date_publication, echeance
ORDER BY date_publication, echeance
""")

display(evolution.head())

for echeance in ["J1", "J2"]:
    subset = evolution[evolution["echeance"] == echeance]
    plt.plot(
        subset["date_publication"],
        subset["niveau_moyen"],
        label=echeance,
    )

plt.title("Évolution du niveau moyen de danger")
plt.ylabel("Niveau moyen")
plt.xlabel("")
plt.legend()
plt.show()


Le niveau moyen permet de voir les périodes où le danger augmente à l'échelle nationale. Une hausse ponctuelle peut correspondre à un épisode chaud et sec, mais cette analyse ne cherche pas encore à expliquer la cause météo.


## 7. Danger selon le mois


In [ ]:
par_mois = lire_requete(f"""
SELECT
  EXTRACT(MONTH FROM date_publication) AS mois,
  echeance,
  ROUND(AVG(niveau_danger), 2) AS niveau_moyen,
  ROUND(100 * COUNTIF(niveau_danger >= 3) / COUNT(*), 2) AS pct_danger_eleve
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY mois, echeance
ORDER BY mois, echeance
""")

display(par_mois)

for echeance in ["J1", "J2"]:
    subset = par_mois[par_mois["echeance"] == echeance]
    plt.plot(subset["mois"], subset["pct_danger_eleve"], marker="o", label=echeance)

plt.title("Part des niveaux 3 et 4 selon le mois")
plt.ylabel("Danger élevé ou très élevé (%)")
plt.xlabel("Mois")
plt.legend()
plt.show()


Cette vue résume la saisonnalité du danger. Les mois les plus chauds devraient concentrer davantage de niveaux 3 et 4, mais l'intensité varie selon les années.


## 8. Départements les plus souvent concernés par un danger élevé


In [ ]:
par_departement = lire_requete(f"""
SELECT
  f.numero_departement,
  ANY_VALUE(d.departement) AS departement,
  COUNT(*) AS observations,
  ROUND(AVG(f.niveau_danger), 2) AS niveau_moyen,
  ROUND(100 * COUNTIF(f.niveau_danger >= 3) / COUNT(*), 2) AS pct_danger_eleve,
  COUNTIF(f.niveau_danger = 4) AS nb_tres_eleve
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie` AS f
LEFT JOIN `{PROJET_GCP}.{DATASET}.dim_departement` AS d
  ON f.numero_departement = d.numero_departement
GROUP BY f.numero_departement
ORDER BY pct_danger_eleve DESC
""")

display(par_departement.head(15))

par_departement.head(15).sort_values("pct_danger_eleve").plot(
    x="departement",
    y="pct_danger_eleve",
    kind="barh",
    figsize=(9, 6),
    legend=False,
)
plt.title("Départements avec le plus de niveaux 3 ou 4")
plt.xlabel("Part des observations (%)")
plt.ylabel("")
plt.show()


Cette comparaison met en évidence les territoires qui reçoivent le plus souvent un niveau 3 ou 4 pendant la période disponible. Il faut garder en tête que la période observée est courte par rapport à un historique climatique complet.


## 9. Journées avec le plus de départements en danger élevé


In [ ]:
jours_forts = lire_requete(f"""
SELECT
  date_publication,
  echeance,
  COUNTIF(niveau_danger >= 3) AS departements_niveau_3_ou_4,
  COUNTIF(niveau_danger = 4) AS departements_niveau_4,
  ROUND(AVG(niveau_danger), 2) AS niveau_moyen
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY date_publication, echeance
ORDER BY departements_niveau_3_ou_4 DESC, date_publication
LIMIT 20
""")

jours_forts


Ces dates correspondent aux épisodes où le danger est le plus généralisé en France. Elles sont intéressantes à comparer ensuite avec la météo de la même période.


## 10. Comparaison directe de J1 avec J2


In [ ]:
comparaison_j1_j2 = lire_requete(f"""
WITH j1 AS (
  SELECT
    reference_time,
    numero_departement,
    niveau_danger AS niveau_j1
  FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
  WHERE echeance = 'J1'
),
j2 AS (
  SELECT
    reference_time,
    numero_departement,
    niveau_danger AS niveau_j2
  FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
  WHERE echeance = 'J2'
)
SELECT
  j1.niveau_j1,
  j2.niveau_j2,
  COUNT(*) AS observations
FROM j1
INNER JOIN j2
  USING (reference_time, numero_departement)
GROUP BY niveau_j1, niveau_j2
ORDER BY niveau_j1, niveau_j2
""")

display(comparaison_j1_j2)

matrice = comparaison_j1_j2.pivot(
    index="niveau_j1",
    columns="niveau_j2",
    values="observations",
).fillna(0)

display(matrice)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrice)
ax.set_xticks(range(len(matrice.columns)))
ax.set_xticklabels(matrice.columns)
ax.set_yticks(range(len(matrice.index)))
ax.set_yticklabels(matrice.index)
ax.set_xlabel("Niveau J2")
ax.set_ylabel("Niveau J1")
plt.colorbar(image, ax=ax)
plt.title("Passage du niveau J1 au niveau J2")
plt.show()


La diagonale représente les départements dont le niveau reste identique entre J1 et J2. Les cases autour montrent les hausses ou baisses prévues d'un jour à l'autre.


## 11. Fréquence des hausses et baisses entre J1 et J2


In [ ]:
variations = lire_requete(f"""
WITH niveaux AS (
  SELECT
    reference_time,
    numero_departement,
    MAX(IF(echeance = 'J1', niveau_danger, NULL)) AS niveau_j1,
    MAX(IF(echeance = 'J2', niveau_danger, NULL)) AS niveau_j2
  FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
  GROUP BY reference_time, numero_departement
)
SELECT
  CASE
    WHEN niveau_j2 > niveau_j1 THEN 'Hausse'
    WHEN niveau_j2 < niveau_j1 THEN 'Baisse'
    ELSE 'Stable'
  END AS evolution,
  COUNT(*) AS observations,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM niveaux
WHERE niveau_j1 IS NOT NULL
  AND niveau_j2 IS NOT NULL
GROUP BY evolution
ORDER BY observations DESC
""")

display(variations)

variations.plot(
    x="evolution",
    y="pct",
    kind="bar",
    figsize=(7, 4),
    legend=False,
)
plt.title("Évolution entre J1 et J2")
plt.ylabel("Part des observations (%)")
plt.xlabel("")
plt.show()


Cette dernière comparaison montre simplement si la prévision reste le plus souvent stable ou si elle change souvent entre J1 et J2. C'est un bon indicateur de la variabilité du danger annoncé.


## Bilan

Cet EDA permet surtout de vérifier :

- que les publications sont complètes ;
- quels niveaux de danger sont les plus fréquents ;
- quand les niveaux élevés apparaissent ;
- quels départements sont les plus souvent concernés ;
- comment les prévisions changent entre J1 et J2.

Le niveau Météo-France reste une **prévision de danger**. Il ne correspond pas directement au nombre de départs de feu réellement observés.
